# Birch–Murnaghan $f$–$F$ diagnostic

A normalized-stress plot is a compact visual check of the order required by a Birch–Murnaghan equation of state. It plots Eulerian strain

$$f=\frac{1}{2}\left[\left(\frac{V_0}{V}\right)^{2/3}-1\right]$$

against normalized stress

$$F=\frac{P}{3f(1+2f)^{5/2}}.$$

For BM3, $F=K_0+\tfrac{3}{2}K_0(K_0'-4)f$. BM2 fixes $K_0'=4$ and is therefore horizontal. This notebook uses the bundled room-temperature coesite P–V observations of Levien and Prewitt (1981); no thermal model or thermal correction is involved.

In [ ]:
from importlib import resources

import matplotlib.pyplot as plt
import numpy as np

from peritheos import get_material_document, material_from_dict
from peritheos.diagnostics import birch_murnaghan_finite_strain_diagnostic
from peritheos.eos.rt import BM2, BM3
from peritheos.fitting import fit_rt_eos

plt.style.use("seaborn-v0_8-whitegrid")

material = material_from_dict(get_material_document("coesite"))
record = material.get_eos_record("coesite_levien_1981_bm3_1")
dataset = next(
    item
    for item in material.datasets
    if item["identifier"] == "coesite_levien_1981_table7_pv"
)
resource = resources.files("peritheos.data").joinpath(dataset["resource"]["path"])
with resources.as_file(resource) as data_path:
    data = np.genfromtxt(data_path, delimiter=",", names=True)

pressure = data["pressure_kbar"] / 10.0
pressure_sigma = data["pressure_sigma_kbar"] / 10.0
volume = data["volume_a3"]
volume_sigma = data["volume_sigma_a3"]
reference_volume = record.eos.V0

print(f"{material.name}: {len(data)} room-temperature observations")
print(f"Bundled source: {dataset['source_location']}")
print(f"Fixed reference volume: {reference_volume:.4f} Å³")

## Fit BM2 and BM3 consistently

The publication reports an unweighted BM3 fit, so both comparisons below minimize the same unweighted pressure residuals. The independently measured ambient $V_0$ is fixed for both models. This leaves one free coefficient in BM2 ($K_0$) and two in BM3 ($K_0$ and $K_0'$).

In [ ]:
bm2_fit = fit_rt_eos(
    BM2,
    volume,
    pressure,
    initial={"K0": 100.0},
    fixed={"V0": reference_volume},
)
bm3_fit = fit_rt_eos(
    BM3,
    volume,
    pressure,
    initial={"K0": 100.0, "K0_prime": 6.0},
    fixed={"V0": reference_volume},
)

print(f"BM2: K0 = {bm2_fit.parameters['K0']:.2f} GPa, AIC = {bm2_fit.aic:.2f}")
print(
    f"BM3: K0 = {bm3_fit.parameters['K0']:.2f} ± "
    f"{bm3_fit.standard_errors['K0']:.2f} GPa, "
    f"K0' = {bm3_fit.parameters['K0_prime']:.2f} ± "
    f"{bm3_fit.standard_errors['K0_prime']:.2f}, "
    f"AIC = {bm3_fit.aic:.2f}"
)

## Transform the compression observations

The nearly ambient row is excluded from the visualization. As $f\rightarrow0$, both the numerator and denominator defining $F$ approach zero, so experimental noise is strongly amplified. The fit above still uses every reported observation. The diagnostic function supplies the transformed values and first-order marginal error bars; plotting remains ordinary notebook code.

In [ ]:
compression = pressure > 0.1
diagnostic = birch_murnaghan_finite_strain_diagnostic(
    volume[compression],
    pressure[compression],
    model=bm3_fit.model,
    pressure_sigma=pressure_sigma[compression],
    volume_sigma=volume_sigma[compression],
)
bm2_diagnostic = birch_murnaghan_finite_strain_diagnostic(
    volume[compression],
    pressure[compression],
    model=bm2_fit.model,
)

f_bm2, F_bm2 = bm2_diagnostic.model_curve()
f_bm3, F_bm3 = diagnostic.model_curve()

In [ ]:
blue = "#0072B2"
vermillion = "#D55E00"

fig, (ax_pv, ax_ff) = plt.subplots(1, 2, figsize=(11.0, 4.4), layout="constrained")

# The original P–V observations and both fitted equations.
ax_pv.errorbar(
    volume,
    pressure,
    xerr=volume_sigma,
    yerr=pressure_sigma,
    fmt="o",
    ms=5,
    color="black",
    ecolor="0.55",
    capsize=2,
    label="Levien & Prewitt (1981)",
    zorder=3,
)
volume_grid = np.linspace(volume.min(), reference_volume, 400)
ax_pv.plot(
    volume_grid,
    bm2_fit.model.pressure(volume_grid),
    color=blue,
    linestyle="--",
    linewidth=2,
    label="BM2",
)
ax_pv.plot(
    volume_grid,
    bm3_fit.model.pressure(volume_grid),
    color=vermillion,
    linewidth=2,
    label="BM3",
)
ax_pv.set(
    xlabel=r"Conventional-cell volume ($\mathrm{Å^3}$)",
    ylabel="Pressure (GPa)",
    title="Room-temperature coesite compression",
)
ax_pv.legend(frameon=False)

# The f–F diagnostic is plotted explicitly from the numerical result.
ax_ff.errorbar(
    diagnostic.strain,
    diagnostic.normalized_stress,
    xerr=diagnostic.strain_standard_error,
    yerr=diagnostic.normalized_stress_standard_error,
    fmt="o",
    ms=5,
    markerfacecolor="white",
    markeredgecolor="black",
    ecolor="0.55",
    capsize=2,
    label="observations",
    zorder=3,
)
ax_ff.plot(
    f_bm2, F_bm2, color=blue, linestyle="--", linewidth=2, label=r"BM2: $K_0'=4$"
)
ax_ff.plot(
    f_bm3,
    F_bm3,
    color=vermillion,
    linewidth=2,
    label=rf"BM3: $K_0'={bm3_fit.parameters['K0_prime']:.2f}$",
)
ax_ff.set(
    xlabel=r"Eulerian strain, $f$",
    ylabel=r"Normalized stress, $F$ (GPa)",
    title=r"Birch–Murnaghan $f$–$F$ diagnostic",
)
ax_ff.legend(frameon=False)

for ax in (ax_pv, ax_ff):
    ax.grid(True, color="0.9", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)

plt.show()

## Interpretation

The transformed observations rise with $f$, whereas BM2 requires a horizontal line. BM3 captures that trend through $K_0'>4$. The plot is a diagnostic rather than a replacement for regression: $V_0$ is treated as fixed, transformed errors are heteroscedastic and correlated, and the same observations should not be refitted by unweighted linear regression in $f$–$F$ space. A quantitative order decision should also consider the uncertainty of $K_0'$, residual structure, and like-for-like BM2/BM3 fit statistics.